# Brent forecasting with a regime-dependent LASSO

One model: a LASSO regression of the cumulative Brent futures log return 1, 5, 10, 20, 30, 40, 50 and 60
trading days ahead (one model per horizon, direct forecasts), in which every predictor can have a
different effect in calm and turbulent markets. Forecasts are converted to prices.
Benchmarks required by the brief: no-change (RW), historical mean, AR(1).

**Regimes.** A two-state Markov-switching model of daily Brent returns classifies each day as calm or
turbulent. Each regime must last at least 120 trading days. The regime indicator enters the LASSO as an
interaction with every predictor; the LASSO penalty decides whether a regime-specific effect is needed.

**Sample split**

| Block | Dates | Role |
|---|---|---|
| In-sample | each fit's 1,260-day window (about 5 years) | training; the LASSO penalty is chosen by time-ordered cross-validation inside the window |
| Out-of-sample backtest | forecast origins 2016-01-01 to 2026-09-30 | evaluation; every forecast uses only data up to its origin |
| Live test | origins from 2026-10-01 | untouched test period; forecasts logged daily |

**Run.** Open from `OilForecastQF`, run all cells (a few minutes). Results go to `Outputs/lasso/`.

In [ ]:
from dataclasses import dataclass, replace
from pathlib import Path
import inspect
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LassoCV
from sklearn.model_selection import TimeSeriesSplit
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
PROJECT_ROOT = next((p.resolve() for p in (Path.cwd(), Path.cwd() / "OilForecastQF", Path.cwd().parent)
                     if (p / "Data" / "raw" / "yahoo_daily.csv").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the notebook from OilForecastQF (needs Data/raw/yahoo_daily.csv).")
RUN_LEAK_CHECK = False          # True: re-run a short backtest on scrambled future data (about 1 minute)
pd.set_option("display.width", 180)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#E6E8EB", "font.size": 10})

## 1. Settings

In [ ]:
@dataclass
class Config:
    data_dir: Path = PROJECT_ROOT / "Data" / "raw"
    output_dir: Path = PROJECT_ROOT / "Outputs" / "lasso"
    target: str = "brent_fut"
    horizons: tuple = (1, 5, 10, 20, 30, 40, 50, 60)   # one direct model per horizon (trading days)
    oos_start: str = "2016-01-01"      # first out-of-sample forecast origin
    test_start: str = "2026-10-01"     # live test period
    window: int = 1260                 # rolling training window (trading days, about 5 years)
    refit_every: int = 21              # re-estimate every 21 trading days (about monthly)
    ffill_limit: int = 3
    fred_lag: int = 1                  # FRED daily series enter one observation late
    # LASSO
    cv_folds: int = 5                  # time-ordered cross-validation folds inside the window
    n_alphas: int = 30                 # penalty grid
    lasso_eps: float = 1e-2            # smallest penalty = 1% of the largest
    shrink: float = 0.5                # final forecast = 0.5 x LASSO (equal-weight combination with no change)
    # regimes
    min_duration: int = 120            # each regime expected to last >= 120 days, and labels kept >= 120 days
    enter_p: float = 0.7               # label: switch to turbulent if P(turbulent) > 0.7 ...
    exit_p: float = 0.3                # ... back to calm if P(turbulent) < 0.3 ...
    confirm: int = 10                  # ... for 10 consecutive days
    use_season: bool = False
    plot_h: int = 20
    path_years: int = 4
    cost_bps: float = 5.0
    target_vol: float = 25.0
    max_leverage: float = 2.0
    trade_h: int = 20                  # trading rule: use the 20-day forecast and hold for 20 days


cfg = Config()
cfg.output_dir.mkdir(parents=True, exist_ok=True)

## 2. Data and features

Sources: `Data/raw/yahoo_daily.csv`; `Data/raw/fred_daily.csv` if present (lagged one observation).
The calendar is the Brent trading calendar; other series are forward-filled for at most 3 days;
non-positive prices are set to missing.

About 20 candidate predictors, chosen on economic grounds, all known at the close of day $t$.
Every feature is a change or a z-score against its own past year, $z_t=(v_t-\text{mean}_{252})/\text{sd}_{252}$,
so that values in 2020 or 2022 stay comparable with the training window.

| Block | Features | Idea |
|---|---|---|
| Brent momentum and trend | log returns over 1, 5, 20, 60 days; log gap to 50- and 200-day averages | momentum, reversal, trend |
| Volatility | 20/60-day realized volatility ratio; z-scores of 60-day realized volatility and VIX | risk |
| Oil market | z-scores of Brent–WTI spread and 3-2-1 crack spread; crack spread 20-day change | regional balance, refining margin |
| Cross-market (20-day log return) | natural gas, XLE, S&P 500, dollar index, gold, copper, and USD/CAD, USD/NOK, EEM if present | energy equities, risk appetite, dollar, industrial demand |
| Macro (FRED, if present) | 10Y–3M spread 60-day change; OVX z-score; 10Y breakeven 20-day change | business cycle, oil risk, inflation |

In [ ]:
def load_data(cfg):
    raw = pd.read_csv(cfg.data_dir / "yahoo_daily.csv", parse_dates=["date"]).set_index("date").sort_index()
    raw = raw[~raw.index.duplicated(keep="last")].select_dtypes("number")
    raw = raw.where(raw > 0)
    price = raw[cfg.target].dropna()
    other = raw.drop(columns=[cfg.target])
    fred_path = cfg.data_dir / "fred_daily.csv"
    if fred_path.is_file():
        fred = pd.read_csv(fred_path, parse_dates=["date"]).set_index("date").sort_index().select_dtypes("number")
        keep = [c for c in ("t10y3m", "ovx", "breakeven10y") if c in fred]
        fred = fred[keep].dropna(how="all").shift(cfg.fred_lag)       # published the next day
        other = other.join(fred, how="outer")
    other = other.reindex(other.index.union(price.index)).ffill(limit=cfg.ffill_limit).reindex(price.index)
    return price, other


def zscore(s, n=252):
    roll = s.rolling(n, min_periods=int(0.8 * n))
    return (s - roll.mean()) / roll.std()


def build_features(price, other, cfg):
    lp = np.log(price)
    r = lp.diff()
    vol = pd.DataFrame({"rv20": r.rolling(20, min_periods=15).std() * np.sqrt(252),
                        "rv60": r.rolling(60, min_periods=45).std() * np.sqrt(252)})
    f = {f"brent_ret{k}": lp.diff(k) for k in (1, 5, 20, 60)}
    f["brent_gap_ma50"] = lp - np.log(price.rolling(50, min_periods=40).mean())
    f["brent_gap_ma200"] = lp - np.log(price.rolling(200, min_periods=160).mean())
    f["rv_ratio"] = vol["rv20"] / vol["rv60"]
    f["rv60_z"] = zscore(vol["rv60"])
    if "vix" in other:
        f["vix_z"] = zscore(other["vix"])
    if "wti_fut" in other:
        f["brent_wti_z"] = zscore(lp - np.log(other["wti_fut"]))
    if {"gasoline", "heating_oil", "wti_fut"} <= set(other.columns):
        cl = other["wti_fut"]
        crack = ((2 * 42 * other["gasoline"] + 42 * other["heating_oil"] - 3 * cl) / 3 / cl).where(lambda s: s.abs() < 2)
        f["crack_z"], f["crack_d20"] = zscore(crack), crack.diff(20)
    for c in ("natgas", "xle", "sp500", "dxy", "gold", "copper", "usdcad", "usdnok", "eem"):
        if c in other and other[c].notna().sum() > 500:
            f[f"{c}_ret20"] = np.log(other[c]).diff(20)
    if "t10y3m" in other:
        f["t10y3m_d60"] = other["t10y3m"].diff(60)
    if "ovx" in other:
        f["ovx_z"] = zscore(other["ovx"])
    if "breakeven10y" in other:
        f["breakeven_d20"] = other["breakeven10y"].diff(20)
    if cfg.use_season:
        doy = price.index.dayofyear.to_numpy()
        f["season_sin"] = pd.Series(np.sin(2 * np.pi * doy / 365.25), index=price.index)
        f["season_cos"] = pd.Series(np.cos(2 * np.pi * doy / 365.25), index=price.index)
    X = pd.DataFrame(f, index=price.index).replace([np.inf, -np.inf], np.nan)
    Y = {h: 100 * (lp.shift(-h) - lp) for h in cfg.horizons}         # cumulative h-day log return, %
    return X, Y, lp, vol


price, other = load_data(cfg)
X, Y, logp, VOL = build_features(price, other, cfg)
pd.concat([price.rename("brent_fut"), X], axis=1).rename_axis("date").to_csv(cfg.output_dir / "model_panel.csv")
display(pd.DataFrame({"first date": [price.index[0].date()], "last date": [price.index[-1].date()],
                      "Brent observations": [len(price)], "number of features": [X.shape[1]]}).T.rename(columns={0: ""}))
display(Markdown("**Features:** " + ", ".join(f"`{c}`" for c in X.columns)))

## 3. Model

### 3.1 Regimes: Markov-switching model of daily returns

$$r_t = \mu_{s_t} + \sigma_{s_t}\varepsilon_t,\qquad s_t\in\{\text{calm},\text{turbulent}\},\qquad P(s_t=j\mid s_{t-1}=i)=p_{ij},\qquad p_{jj}\ge 1-\tfrac{1}{120}$$

- Estimated by EM (Hamilton filter and Kim smoother) on the same rolling window as the LASSO. Regimes are ordered by volatility, so "turbulent" is always the high-volatility state.
- The constraint on $p_{jj}$ makes each regime last at least 120 trading days on average ($1/(1-p_{jj})$), so a single large return cannot flip the regime.
- The filtered probability $P(\text{turbulent}\mid \text{returns up to } t)$ uses only past data.
- **Regime label $d_t$** (1 = turbulent): switches to turbulent when the probability stays above 0.7 for 10 days, back to calm when it stays below 0.3 for 10 days, and never before the current regime has lasted 120 trading days. The rule is causal.

### 3.2 Regime-dependent LASSO

For each horizon $h$, with $z_t$ the standardized predictors:

$$y_{t,h} = 100\,(\log P_{t+h}-\log P_t) = \beta^\top z_t + \gamma^\top (z_t\, d_t) + \delta\, d_t + e_{t,h}$$

$$\min_{\beta,\gamma,\delta}\ \tfrac{1}{2n}\sum_t \big(y_{t,h}-\beta^\top z_t-\gamma^\top z_t d_t-\delta d_t\big)^2+\lambda\big(\lVert\beta\rVert_1+\lVert\gamma\rVert_1+|\delta|\big)$$

- Effect of a predictor in calm markets: $\beta_k$; in turbulent markets: $\beta_k+\gamma_k$. If $\gamma_k=0$ the effect is the same in both regimes, so the LASSO only keeps a regime difference when the data support it.
- $\lambda$: chosen by 5-fold time-ordered cross-validation inside the window, with a gap of $h$ days between training and validation folds (overlapping $h$-day returns would otherwise leak).
- No intercept: with no selected predictor the forecast equals the no-change forecast.
- **Final forecast** $\hat y_{t,h} = 0.5\times$ LASSO, an equal-weight combination with the no-change forecast. With $f$ the LASSO forecast, $\text{MSE}(\lambda f)=\text{MSE}(0)-2\lambda E[fy]+\lambda^2E[f^2]$: when the signal is weak, halving the forecast cuts error from noise faster than it loses signal. The unshrunk forecast is reported as `LASSO_raw`.
- Intervals: errors in the window divided by the 20-day realized volatility scaled to $h$ days; their 2.5/10/90/97.5% quantiles multiplied by today's volatility.

All settings are fixed in advance and not tuned on the backtest.

In [ ]:
try:
    from numba import njit
except Exception:                                   # missing or broken numba: plain Python (slower)
    def njit(*a, **k):
        return a[0] if a and callable(a[0]) else (lambda f: f)


@njit(cache=False)
def _ms_filter(y, mu, sig, P, xi0):
    # pred[t] = P(s_t | data before t), filt[t] = P(s_t | data up to t)
    T = y.shape[0]
    filt = np.empty((T, 2)); pred = np.empty((T, 2))
    p0, p1, ll = xi0[0], xi0[1], 0.0
    c = 1.0 / np.sqrt(2.0 * np.pi)
    for t in range(T):
        pred[t, 0] = p0; pred[t, 1] = p1
        z0 = (y[t] - mu[0]) / sig[0]; z1 = (y[t] - mu[1]) / sig[1]
        a0 = p0 * c / sig[0] * np.exp(-0.5 * z0 * z0)
        a1 = p1 * c / sig[1] * np.exp(-0.5 * z1 * z1)
        s = max(a0 + a1, 1e-300)
        ll += np.log(s)
        g0 = a0 / s; g1 = a1 / s
        filt[t, 0] = g0; filt[t, 1] = g1
        p0 = g0 * P[0, 0] + g1 * P[1, 0]
        p1 = 1.0 - p0
    return filt, pred, ll


@njit(cache=False)
def _ms_smooth(filt, pred, P):
    T = filt.shape[0]
    sm = np.empty((T, 2)); xs = np.zeros((2, 2))
    sm[T - 1, 0] = filt[T - 1, 0]; sm[T - 1, 1] = filt[T - 1, 1]
    for t in range(T - 2, -1, -1):
        r0 = sm[t + 1, 0] / max(pred[t + 1, 0], 1e-300)
        r1 = sm[t + 1, 1] / max(pred[t + 1, 1], 1e-300)
        for i in range(2):
            a = filt[t, i] * P[i, 0] * r0
            b = filt[t, i] * P[i, 1] * r1
            sm[t, i] = a + b
            xs[i, 0] += a; xs[i, 1] += b
    return sm, xs


class RegimeModel:
    """Two-state Markov-switching mean/variance model with minimum expected regime length."""

    def __init__(self, min_duration=120, max_iter=200, tol=1e-7):
        self.p_min, self.max_iter, self.tol = 1 - 1 / min_duration, max_iter, tol

    @staticmethod
    def stationary(P):
        den = 2 - P[0, 0] - P[1, 1]
        x0 = (1 - P[1, 1]) / den if den > 1e-12 else 0.5
        return np.array([x0, 1 - x0])

    def _em(self, y, mu, sig, P):
        floor, prev = 0.05 * y.std(), -np.inf
        for _ in range(self.max_iter):
            filt, pred, ll = _ms_filter(y, mu, sig, P, self.stationary(P))
            sm, xs = _ms_smooth(filt, pred, P)
            for j in range(2):
                w = sm[:, j] + 1e-10
                mu[j] = np.sum(w * y) / w.sum()
                sig[j] = max(np.sqrt(np.sum(w * (y - mu[j]) ** 2) / w.sum()), floor)
            P = xs / xs.sum(1, keepdims=True)
            for j in range(2):
                if P[j, j] < self.p_min:
                    P[j, j], P[j, 1 - j] = self.p_min, 1 - self.p_min
            if ll - prev < self.tol * abs(ll):
                break
            prev = ll
        return mu, sig, P, _ms_filter(y, mu, sig, P, self.stationary(P))[2]

    def fit(self, y, warm=None):
        vol = pd.Series(y).rolling(20, min_periods=5).std().bfill().to_numpy()
        g = vol > np.median(vol)
        d = max(0.995, self.p_min)
        starts = [(np.array([y[~g].mean(), y[g].mean()]), np.array([y[~g].std(), y[g].std()]),
                   np.array([[d, 1 - d], [1 - d, d]]))]
        if warm is not None:
            starts.append((warm.mu.copy(), warm.sig.copy(), warm.P.copy()))
        mu, sig, P, ll = max((self._em(y, *[a.copy() for a in s]) for s in starts), key=lambda res: res[3])
        o = np.argsort(sig)                          # 0 = calm, 1 = turbulent
        self.mu, self.sig, self.P, self.loglik = mu[o], sig[o], P[o][:, o], ll
        return self

    def p_turbulent(self, y):
        return _ms_filter(y, self.mu, self.sig, self.P, self.stationary(self.P))[0][:, 1]


class RegimeLabel:
    """Causal label, run day by day through the whole sample: switch only after `confirm` days beyond the
    threshold, and never before the current regime has lasted `min_duration` days."""

    def __init__(self, cfg):
        self.cfg, self.state, self.run, self.since = cfg, None, 0, cfg.min_duration

    def step(self, v):
        c = self.cfg
        if self.state is None:
            self.state = int(v > 0.5)
        self.since += 1
        cross = v > c.enter_p if self.state == 0 else v < c.exit_p
        self.run = self.run + 1 if cross else 0
        if self.run >= c.confirm and self.since >= c.min_duration:
            self.state, self.run, self.since = 1 - self.state, 0, 0
        return self.state


ALPHA_ARG = "n_alphas" if "n_alphas" in inspect.signature(LassoCV).parameters else "alphas"


def design(A, lab, train):
    """[z, z*d, d], standardized with the training rows."""
    mu, sd = np.nanmean(A[train], 0), np.nanstd(A[train], 0)
    Z = np.nan_to_num((A - mu) / np.where(sd > 0, sd, 1.0))
    D = np.column_stack([Z, Z * lab[:, None], lab])
    m, s = D[train].mean(0), D[train].std(0)
    return np.where(s > 0, (D - m) / np.where(s > 0, s, 1.0), 0.0)


def fit_lasso(D, y, rows, h, cfg):
    rows = rows[np.isfinite(y[rows])]
    m = LassoCV(cv=TimeSeriesSplit(n_splits=cfg.cv_folds, gap=h), eps=cfg.lasso_eps, fit_intercept=False,
                max_iter=5000, tol=1e-3, **{ALPHA_ARG: cfg.n_alphas})
    m.fit(D[rows], y[rows])
    return m, rows

## 4. Rolling estimation and forecasts

- **Rolling window:** every 21 trading days (about monthly) from 2016-01-01, the regime model and the LASSO are re-estimated on the previous 1,260 trading days (about 5 years). For horizon $h$, the last training row is $t-h$, so every target is already observed at the forecast date.
- **Training / validation:** inside each window, $\lambda$ is chosen by time-ordered cross-validation with an $h$-day gap. Nothing after the forecast date is used.
- **Daily update:** between re-estimations the parameters stay fixed; every day the regime probability and label are updated with the new return and the forecast uses that day's features.
- **Benchmarks** on the same dates and window: RW (no change), HistMean (window mean h-day return), AR(1) on daily returns, iterated.
- **In-sample forecasts:** the latest model of each horizon applied to its own training rows.

In [ ]:
def run_backtest(X, Y, lp, vol, cfg, verbose=True):
    dates, N, names = X.index, len(X), list(X.columns)
    A = X.to_numpy(float)
    rr = 100 * lp.diff().to_numpy()
    rv = vol["rv20"].to_numpy()
    start = max(int(dates.searchsorted(pd.Timestamp(cfg.oos_start))), cfg.window + 260)
    refits = list(range(start, N, cfg.refit_every))
    out = {h: {k: np.full(N, np.nan) for k in ("LASSO", "LASSO_raw", "q025", "q10", "q90", "q975", "HistMean", "AR")}
           for h in cfg.horizons}
    p_turb, lab_out = np.full(N, np.nan), np.full(N, np.nan)
    machine = RegimeLabel(cfg)
    coefs, reg_log, ins, rmodel, t0 = [], [], {}, None, time.time()
    for k, i0 in enumerate(refits):
        i1 = refits[k + 1] if k + 1 < len(refits) else N
        lo = i0 - cfg.window
        rmodel = RegimeModel(cfg.min_duration).fit(rr[lo + 1:i0 + 1], warm=rmodel)    # returns known at i0
        p = rmodel.p_turbulent(np.nan_to_num(rr[lo:i1]))      # row t uses returns up to t only
        first = lo if k == 0 else i0                          # the first window labels its own history once
        p_turb[first:i1] = p[first - lo:]
        for t in range(first, i1):                            # one continuous label path, day by day
            lab_out[t] = machine.step(p_turb[t])
        lab = lab_out[lo:i1]
        reg_log.append({"refit": dates[i0], "vol_calm_ann_%": rmodel.sig[0] * np.sqrt(252),
                        "vol_turb_ann_%": rmodel.sig[1] * np.sqrt(252), "p_stay_calm": rmodel.P[0, 0],
                        "p_stay_turb": rmodel.P[1, 1], "turbulent_share_window": np.mean(lab[:i0 - lo])})
        D = design(A[lo:i1], lab, np.arange(0, i0 - lo))
        w = rr[lo + 1:i0 + 1]
        c, phi = np.polynomial.polynomial.polyfit(w[:-1], w[1:], 1)
        for h in cfg.horizons:
            y = Y[h].to_numpy()[lo:i1]
            m, rows = fit_lasso(D, y, np.arange(0, i0 - lo - h + 1), h, cfg)
            o = out[h]
            o["LASSO_raw"][i0:i1] = m.predict(D[i0 - lo:])
            o["LASSO"][i0:i1] = cfg.shrink * o["LASSO_raw"][i0:i1]
            scale = 100 * rv[lo:i1] * np.sqrt(h / 252)
            z = (y[rows] - cfg.shrink * m.predict(D[rows])) / scale[rows]
            q = np.nanquantile(z, [0.025, 0.10, 0.90, 0.975])
            for name, qq in zip(("q025", "q10", "q90", "q975"), q):
                o[name][i0:i1] = qq * scale[i0 - lo:]
            o["HistMean"][i0:i1] = np.nanmean(y[:i0 - lo - h + 1])
            cur, acc = rr[i0:i1].copy(), np.zeros(i1 - i0)
            for _ in range(h):
                cur = c + phi * cur
                acc = acc + cur
            o["AR"][i0:i1] = acc
            coefs.append({"refit": dates[i0], "h": h, "lambda": m.alpha_,
                          **{f"base:{n}": b for n, b in zip(names, m.coef_[:len(names)])},
                          **{f"turb:{n}": b for n, b in zip(names, m.coef_[len(names):2 * len(names)])},
                          "turb:regime_dummy": m.coef_[-1]})
            if k == len(refits) - 1:
                ins[h] = {"model": m, "rows": lo + rows, "D": D, "lo": lo, "q": q}
    if verbose:
        print(f"{len(refits)} re-estimations x {len(cfg.horizons)} horizons, {time.time() - t0:.0f}s. "
              f"First origin {dates[start].date()}, last {dates[-1].date()}.")
    return (out, pd.DataFrame(coefs), pd.DataFrame(reg_log).set_index("refit"),
            pd.Series(p_turb, index=dates), pd.Series(lab_out, index=dates), ins)


out, coefs, reg_log, p_turb_all, label, ins = run_backtest(X, Y, logp, VOL, cfg)
p_turb = p_turb_all.where(out[cfg.horizons[0]]["LASSO"] == out[cfg.horizons[0]]["LASSO"])   # forecast origins only
coefs.to_csv(cfg.output_dir / "lasso_coefficients.csv", index=False)
reg_log.to_csv(cfg.output_dir / "regime_parameters.csv")

In [ ]:
def to_long(cols_by_h, lp, rows_by_h, sample=None):
    dates, N, P, L = lp.index, len(lp), np.exp(lp.to_numpy()), lp.to_numpy()
    recs = []
    for h, rows in rows_by_h.items():
        t = np.asarray(rows)
        tgt = t + h
        has = tgt < N
        d = pd.DataFrame({"origin": dates[t], "h": h,
                          "target_date": [dates[j] if j < N else pd.NaT for j in tgt], "P_t": P[t],
                          "y": np.where(has, L[np.minimum(tgt, N - 1)] - L[t], np.nan), "RW": 0.0})
        for name, arr in cols_by_h[h].items():
            d[name] = arr[t] / 100
        recs.append(d)
    fc = pd.concat(recs, ignore_index=True)
    fc["P_target"] = fc["P_t"] * np.exp(fc["y"])
    test = pd.Timestamp(cfg.test_start)
    fc["sample"] = sample or np.where(fc["origin"] >= test, "Live test",
                                      np.where(fc["target_date"] < test, "Out-of-sample", "straddles test start"))
    return fc


fc = to_long(out, logp, {h: np.flatnonzero(np.isfinite(out[h]["LASSO"])) for h in cfg.horizons})
ins_cols, ins_rows = {}, {}
for h, d in ins.items():
    arr = {k: np.full(len(X), np.nan) for k in ("LASSO", "LASSO_raw", "q025", "q10", "q90", "q975")}
    arr["LASSO_raw"][d["rows"]] = d["model"].predict(d["D"][d["rows"] - d["lo"]])
    arr["LASSO"][d["rows"]] = cfg.shrink * arr["LASSO_raw"][d["rows"]]
    scale = 100 * VOL["rv20"].to_numpy() * np.sqrt(h / 252)
    for name, v in zip(("q025", "q10", "q90", "q975"), d["q"]):
        arr[name][d["rows"]] = v * scale[d["rows"]]
    ins_cols[h], ins_rows[h] = arr, d["rows"]
fc_in = to_long(ins_cols, logp, ins_rows, sample="In-sample")
fc.to_csv(cfg.output_dir / "forecasts_out_of_sample.csv", index=False)
fc_in.to_csv(cfg.output_dir / "forecasts_in_sample.csv", index=False)
r20 = ins[cfg.plot_h]["rows"]
print(f"In-sample rows of the latest {cfg.plot_h}-day model: {X.index[r20[0]].date()} to {X.index[r20[-1]].date()}")
display(fc.groupby("sample")["origin"].agg(["min", "max", "count"]).rename(columns={"count": "rows (origins x horizons)"}))

## 5. Results

### 5.1 In-sample vs out-of-sample accuracy

Price errors in \$/bbl. **RMSE ratio vs RW** below 1 means the model beats the no-change forecast.
DM p-value: one-sided Diebold–Mariano test (Harvey–Leybourne–Newbold correction, Newey–West variance) that the model has lower squared price error than RW.
Direction: hit rate, and precision / recall / F1 for "price goes up"; `up_share_%` is the share of up moves, the precision a model gets by always saying "up". Coverage: share of outcomes inside the 80% and 95% intervals.

In [ ]:
def dm_pvalue(e_m, e_b, h):
    d = e_m ** 2 - e_b ** 2
    n = len(d)
    if n < 30 or np.allclose(d, 0):
        return np.nan
    u = d - d.mean()
    v = u @ u / n + 2 * sum((1 - k / h) * (u[k:] @ u[:-k]) / n for k in range(1, h))
    if v <= 0:
        return np.nan
    stat = d.mean() / np.sqrt(v / n) * np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    return stats.t.cdf(stat, n - 1)


def evaluate(df, models):
    rows = []
    for (h, smp), g in df.dropna(subset=["y"]).groupby(["h", "sample"]):
        e_rw = g["P_t"] - g["P_target"]
        for m in models:
            if m not in g or g[m].isna().all():
                continue
            e = g["P_t"] * np.exp(g[m]) - g["P_target"]
            row = {"sample": smp, "h": h, "model": m, "n": len(g), "MAE_$": e.abs().mean(),
                   "RMSE_$": np.sqrt((e ** 2).mean()), "MAPE_%": 100 * (e.abs() / g["P_target"]).mean(),
                   "RMSE_ratio_vs_RW": np.sqrt((e ** 2).mean() / (e_rw ** 2).mean())}
            if m != "RW":
                row["DM_p"] = dm_pvalue(e.to_numpy(), e_rw.to_numpy(), h)
                up_hat, up = g[m] > 0, g["y"] > 0
                tp = (up_hat & up).sum()
                prec, rec = tp / max(up_hat.sum(), 1), tp / max(up.sum(), 1)
                row.update({"hit_%": 100 * (up_hat == up).mean(), "up_share_%": 100 * up.mean(),
                            "precision_up": prec, "recall_up": rec,
                            "F1_up": 2 * prec * rec / (prec + rec) if prec + rec > 0 else 0.0})
            if m == "LASSO":
                res = g["y"] - g[m]
                row["cover80_%"] = 100 * ((res >= g["q10"]) & (res <= g["q90"])).mean()
                row["cover95_%"] = 100 * ((res >= g["q025"]) & (res <= g["q975"])).mean()
            rows.append(row)
    return pd.DataFrame(rows)


MODELS = ["RW", "HistMean", "AR", "LASSO_raw", "LASSO"]
met = evaluate(pd.concat([fc_in, fc[fc["sample"].isin(["Out-of-sample", "Live test"])]]), MODELS)
met.to_csv(cfg.output_dir / "metrics.csv", index=False)
lm = met[met["model"] == "LASSO"]
order = [c for c in ["In-sample", "Out-of-sample", "Live test"] if c in set(lm["sample"])]
summary = pd.concat({"RMSE ratio vs RW": lm.pivot(index="h", columns="sample", values="RMSE_ratio_vs_RW")[order],
                     "hit rate %": lm.pivot(index="h", columns="sample", values="hit_%")[order]}, axis=1).round(3)
display(Markdown("**LASSO: in-sample vs out-of-sample (price RMSE ratio vs RW, direction hit rate)**"))
display(summary)

In [ ]:
oos = met[met["sample"] == "Out-of-sample"]
display(Markdown("**Out-of-sample, all models: price RMSE ratio vs RW**"))
display(oos.pivot(index="model", columns="h", values="RMSE_ratio_vs_RW").loc[MODELS].round(3))
display(Markdown("**Out-of-sample, LASSO in detail**"))
cols = ["h", "n", "MAE_$", "RMSE_$", "MAPE_%", "RMSE_ratio_vs_RW", "DM_p", "hit_%", "up_share_%", "precision_up",
        "recall_up", "F1_up", "cover80_%", "cover95_%"]
display(oos[oos["model"] == "LASSO"][cols].round(3).set_index("h"))

### 5.2 What the forecasts look like: in-sample and out-of-sample

**Forecast paths.** Every 21 trading days, the forecast path over the eight horizons is drawn from that day's price
(green: model expects a rise over 60 days; red: a fall). Black is the realized price. Shaded: turbulent regime.
Top: in-sample, the latest model on its own training rows. Bottom: out-of-sample, the last few years of rolling forecasts.

**Does the forecast sort the outcomes?** Forecasts are split into five equal groups, from the most bearish (Q1) to the most bullish (Q5).
If the model has signal, the realized return rises from Q1 to Q5. IC is the rank correlation between forecast and outcome.

In [ ]:
def paths(df, origins):
    piv = df.pivot_table(index="origin", columns="h", values="LASSO")
    p0 = df.drop_duplicates("origin").set_index("origin")["P_t"]
    for o in origins:
        if o not in piv.index or piv.loc[o].isna().any():
            continue
        xs = [o] + [o + pd.offsets.BDay(int(h)) for h in piv.columns]
        ys = [p0[o]] + list(p0[o] * np.exp(piv.loc[o].to_numpy()))
        yield xs, ys, ys[-1] > ys[0]


oos_fc = fc[fc["sample"].isin(["Out-of-sample", "Live test", "straddles test start"])]
o_orig = np.sort(oos_fc["origin"].unique())
o_orig = o_orig[o_orig >= o_orig[-1] - np.timedelta64(365 * cfg.path_years, "D")][::-1][::21][::-1]
i_orig = np.sort(fc_in.groupby("origin")["h"].nunique().loc[lambda s: s == len(cfg.horizons)].index)[::21]

fig, axes = plt.subplots(2, 1, figsize=(13, 8.5))
for ax, df, origins, title in [
        (axes[0], fc_in, i_orig, "In-sample: forecast paths of the latest model on its training period"),
        (axes[1], oos_fc, o_orig, f"Out-of-sample: rolling forecast paths, last {cfg.path_years} years")]:
    lo_d, hi_d = pd.Timestamp(origins[0]), pd.Timestamp(origins[-1]) + pd.offsets.BDay(60)
    act = price.loc[lo_d:hi_d]
    ax.plot(act.index, act, color="black", lw=1.0, label="actual price")
    if df is oos_fc:
        lab_ = label.reindex(act.index)
        ax.fill_between(act.index, 0, 1, where=(lab_ == 1).to_numpy(), color="#C0392B", alpha=0.08,
                        transform=ax.get_xaxis_transform(), label="turbulent regime")
        ax.axvspan(pd.Timestamp(cfg.test_start), hi_d, color="#C38728", alpha=0.12, label="live test")
    for xs, ys, up in paths(df, origins):
        ax.plot(xs, ys, color="#1F9D55" if up else "#C0392B", lw=1.3, alpha=0.9)
        ax.plot(xs[0], ys[0], "o", color="#1F9D55" if up else "#C0392B", ms=2.5)
    ax.plot([], [], color="#1F9D55", label="forecast path: up"); ax.plot([], [], color="#C0392B", label="forecast path: down")
    ax.set_title(title, fontsize=11); ax.set_ylabel("$/bbl"); ax.legend(fontsize=8, loc="upper left")
fig.tight_layout(); fig.savefig(cfg.output_dir / "forecast_paths.png", dpi=150); plt.show()

h = cfg.plot_h
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
ic_rows = []
for ax, smp in zip(axes, ["In-sample", "Out-of-sample"]):
    src = fc_in if smp == "In-sample" else fc[fc["sample"] == "Out-of-sample"]
    g = src[src["h"] == h].dropna(subset=["y", "LASSO"]).copy()
    g["Q"] = pd.qcut(g["LASSO"].rank(method="first"), 5, labels=[f"Q{i}" for i in range(1, 6)])
    b = g.groupby("Q", observed=True).agg(forecast=("LASSO", "mean"), realized=("y", "mean")) * 100
    b.plot.bar(ax=ax, color=["#B9C7E8", "#2E4A9E"], rot=0)
    ic = stats.spearmanr(g["LASSO"], g["y"]).correlation
    ic_rows.append({"sample": smp, "h": h, "IC": ic, "Q5 - Q1 realized (%)": b["realized"].iloc[-1] - b["realized"].iloc[0]})
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"{smp}: {h}-day return by forecast quintile (IC = {ic:.2f})", fontsize=10)
    ax.set_xlabel("forecast quintile (Q1 most bearish, Q5 most bullish)"); ax.set_ylabel("mean return, %")
fig.tight_layout(); fig.savefig(cfg.output_dir / "forecast_quintiles.png", dpi=150); plt.show()
display(pd.DataFrame(ic_rows).round(3))

fig, ax = plt.subplots(figsize=(11, 3.8))
allh = met.pivot_table(index="h", columns=["sample", "model"], values="RMSE_ratio_vs_RW")
for (smp, mdl), col_, ls in [(("In-sample", "LASSO"), "#B9C7E8", "-"), (("Out-of-sample", "LASSO"), "#2E4A9E", "-"),
                             (("Out-of-sample", "HistMean"), "#888888", "--"), (("Out-of-sample", "AR"), "#C38728", "--")]:
    if (smp, mdl) in allh:
        ax.plot(allh.index, allh[(smp, mdl)], color=col_, ls=ls, lw=1.6 if mdl == "LASSO" else 1.0, marker="o", ms=4,
                label=f"{mdl}, {smp.lower()}")
ax.axhline(1, color="black", lw=0.8)
ax.set_xticks(list(cfg.horizons)); ax.set_xlabel("horizon (trading days)"); ax.set_ylabel("price RMSE / RW")
ax.set_title("Price RMSE relative to no change, by horizon (below 1 = better)", fontsize=11)
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(cfg.output_dir / "rmse_ratio_by_horizon.png", dpi=150); plt.show()

### 5.3 Regimes and what drives the forecast

Top: Brent with the turbulent regime shaded, and the filtered probability of the turbulent regime (out-of-sample, real time).
Table: regime spells (each at least 120 trading days, except the first, which starts at the first forecast origin, and the current one).

Coefficients of the latest 20-day model (on standardized predictors, % return per 1 s.d.): effect in calm markets $\beta$,
extra effect in turbulent markets $\gamma$, and the total effect in turbulent markets $\beta+\gamma$. Only variables the LASSO kept are shown.
The bar chart shows how often each variable was selected over all re-estimations of the 20-day model.

In [ ]:
pp = p_turb.dropna()
lab = label.reindex(pp.index)
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
pr = price.loc[pp.index]
axes[0].plot(pr.index, pr, color="black", lw=0.8)
axes[0].fill_between(pr.index, 0, 1, where=(lab.reindex(pp.index) == 1).to_numpy(), color="#C0392B", alpha=0.15,
                     transform=axes[0].get_xaxis_transform(), label="turbulent regime (label)")
axes[0].set_ylabel("$/bbl"); axes[0].legend(fontsize=8, loc="upper left")
axes[0].set_title("Brent and the Markov-switching regime (real time, minimum 120 days)", fontsize=11)
axes[1].plot(pp.index, pp, color="#C0392B", lw=0.7)
for v in (cfg.enter_p, cfg.exit_p):
    axes[1].axhline(v, color="grey", ls="--", lw=0.8)
axes[1].set_ylabel("P(turbulent)"); axes[1].set_ylim(0, 1)
fig.tight_layout(); fig.savefig(cfg.output_dir / "regimes.png", dpi=150); plt.show()

sid = (lab != lab.shift()).cumsum()
spells = pd.DataFrame({"regime": lab.groupby(sid).first().map({0.0: "calm", 1.0: "turbulent"}),
                       "start": lab.index.to_series().groupby(sid).first().dt.date,
                       "end": lab.index.to_series().groupby(sid).last().dt.date,
                       "trading_days": lab.groupby(sid).size()}).reset_index(drop=True)
spells.to_csv(cfg.output_dir / "regime_spells.csv", index=False)
last_reg = reg_log.iloc[-1]
display(Markdown(f"**Latest regime model:** annualized volatility calm {last_reg['vol_calm_ann_%']:.0f}%, "
                 f"turbulent {last_reg['vol_turb_ann_%']:.0f}%; expected duration calm "
                 f"{1 / (1 - last_reg['p_stay_calm']):.0f} days, turbulent {1 / (1 - last_reg['p_stay_turb']):.0f} days; "
                 f"current regime: {'turbulent' if lab.iloc[-1] == 1 else 'calm'} (P = {pp.iloc[-1]:.2f})."))
display(spells)

In [ ]:
names = list(X.columns)
last20 = coefs[coefs["h"] == cfg.plot_h].iloc[-1]
ctab = pd.DataFrame({"calm (beta)": [last20[f"base:{n}"] for n in names],
                     "extra in turbulent (gamma)": [last20[f"turb:{n}"] for n in names]}, index=names)
ctab["turbulent (beta + gamma)"] = ctab.sum(axis=1)
ctab = ctab[(ctab.iloc[:, :2] != 0).any(axis=1)]
display(Markdown(f"**Latest {cfg.plot_h}-day model** (refit {last20['refit'].date()}, lambda = {last20['lambda']:.3f}; "
                 f"regime dummy coefficient = {last20['turb:regime_dummy']:.3f})"))
display(ctab.round(3) if len(ctab) else Markdown("No predictor selected: the latest 20-day forecast equals the no-change forecast."))

c20 = coefs[coefs["h"] == cfg.plot_h]
sel = pd.DataFrame({"calm / common effect (beta)": [(c20[f"base:{n}"] != 0).mean() * 100 for n in names],
                    "extra turbulent effect (gamma)": [(c20[f"turb:{n}"] != 0).mean() * 100 for n in names]}, index=names)
sel = sel.loc[sel.sum(axis=1).sort_values(ascending=False).index]
ax = sel.head(15).plot.bar(figsize=(12, 3.5), color=["#2E4A9E", "#C0392B"], rot=60)
ax.set_ylabel("% of re-estimations"); ax.set_ylim(0, 100)
ax.set_title(f"How often the LASSO kept each variable, {cfg.plot_h}-day model ({len(c20)} re-estimations)", fontsize=10)
plt.tight_layout(); plt.savefig(cfg.output_dir / "selection_frequency.png", dpi=150); plt.show()

### 5.4 Robustness

Out-of-sample price RMSE ratio vs RW, split by the regime label at the forecast origin and by year.

In [ ]:
o = fc[fc["sample"] == "Out-of-sample"].dropna(subset=["y"]).copy()
o["regime"] = o["origin"].map(label).map({0.0: "calm", 1.0: "turbulent"})
o["year"] = o["origin"].dt.year


def ratio_by(df, key):
    f = lambda g: np.sqrt(((g["P_t"] * np.exp(g["LASSO"]) - g["P_target"]) ** 2).mean()
                          / ((g["P_t"] - g["P_target"]) ** 2).mean())
    return df.groupby([key, "h"])[["P_t", "LASSO", "P_target"]].apply(f).unstack("h").round(3)


rob_reg, rob_year = ratio_by(o, "regime"), ratio_by(o, "year")
rob_reg.to_csv(cfg.output_dir / "robustness_regime.csv"); rob_year.to_csv(cfg.output_dir / "robustness_year.csv")
display(Markdown("**By regime**")); display(rob_reg)
display(Markdown("**By year**")); display(rob_year)

### 5.5 Economic value

The holding period matches the forecast horizon: every 20 trading days, take the position
$\text{sign}(\hat y_{t,20})\times\min(2,\ 25\%/\sigma^{20d}_t)$ and hold it for 20 days ($\sigma^{20d}$: 20-day realized volatility).
When the LASSO selects no predictor the forecast is zero and the rule holds no position; the hit rate counts only periods with a position.
Costs of 5 bp per unit of change in the position. Benchmarks: buy-and-hold and a long-only position with the same volatility target and rebalancing dates.

In [ ]:
H_TRADE = cfg.trade_h
d1 = fc[(fc["h"] == 1) & fc["sample"].isin(["Out-of-sample", "Live test"])].dropna(subset=["y"]).set_index("origin")["y"]
sig = fc[(fc["h"] == H_TRADE) & fc["sample"].isin(["Out-of-sample", "Live test", "straddles test start"])].set_index("origin")["LASSO"]
sig = sig.reindex(d1.index)
rebal = d1.index[::H_TRADE]
hold = lambda s: s.reindex(rebal).reindex(d1.index).ffill()
lev = np.minimum(cfg.max_leverage, cfg.target_vol / (100 * VOL["rv20"].reindex(d1.index)))
strategies = {"LASSO signal": hold(np.sign(sig) * lev), "Long, vol-targeted": hold(lev),
              "Buy and hold": pd.Series(1.0, index=d1.index)}
rows_e, curves = [], {}
fwd = d1.rolling(H_TRADE).sum().shift(-H_TRADE + 1)
for name, pos in strategies.items():
    pos = pos.fillna(0.0)
    turn = pos.diff().abs().fillna(pos.abs())
    pnl = pos * d1 - cfg.cost_bps / 1e4 * turn
    cum = pnl.cumsum()
    curves[name] = cum
    rows_e.append({"strategy": name, "annual return %": 100 * 252 * pnl.mean(),
                   "annual vol %": 100 * np.sqrt(252) * pnl.std(), "Sharpe": np.sqrt(252) * pnl.mean() / pnl.std(),
                   "max drawdown %": 100 * (np.exp((cum - cum.cummax()).min()) - 1),
                   "turnover per year": 252 * turn.mean(),
                   "invested % of periods": 100 * (pos.reindex(rebal) != 0).mean(),
                   "hit rate of 20-day calls %": 100 * ((np.sign(pos.reindex(rebal)) * fwd.reindex(rebal))
                                                        [pos.reindex(rebal) != 0] > 0).mean()
                   if name == "LASSO signal" else np.nan})
econ = pd.DataFrame(rows_e).set_index("strategy").round(2)
econ.to_csv(cfg.output_dir / "economic_value.csv")
display(econ)
ax = pd.DataFrame(curves).plot(figsize=(11, 3.5), lw=0.9)
ax.set_title(f"Cumulative log return, {H_TRADE}-day holding period (out-of-sample, after costs)", fontsize=11)
plt.tight_layout(); plt.savefig(cfg.output_dir / "equity_curves.png", dpi=150); plt.show()

## 6. Latest forecast

LASSO re-estimated on the most recent window (regime label continued from the backtest), forecast from the last observed close.
Each run appends to `forecast_log.csv`, which is the record for the live test period.

In [ ]:
N = len(X)
P0, d0 = price.iloc[-1], price.index[-1]
A_all, rr_all = X.to_numpy(float), 100 * logp.diff().to_numpy()
lo = N - 1 - cfg.window
lab_live = label.to_numpy()[lo:N]                     # same continuous regime label as the backtest
D_live = design(A_all[lo:N], lab_live, np.arange(0, N - 1 - lo))
live_rows = []
for h in cfg.horizons:
    y = Y[h].to_numpy()[lo:N]
    m, rows = fit_lasso(D_live, y, np.arange(0, N - 1 - lo - h + 1), h, cfg)
    mu = cfg.shrink * float(m.predict(D_live[-1:])[0])
    scale = 100 * VOL["rv20"].to_numpy()[lo:N] * np.sqrt(h / 252)
    q = np.nanquantile((y[rows] - cfg.shrink * m.predict(D_live[rows])) / scale[rows], [0.025, 0.10, 0.90, 0.975])
    q, mu = q * scale[-1] / 100, mu / 100
    live_rows.append({"asof": str(d0.date()), "h": h, "target_date": (d0 + pd.offsets.BDay(h)).date(),
                      "regime": "turbulent" if lab_live[-1] == 1 else "calm",
                      "price_now": P0, "forecast_price": P0 * np.exp(mu),
                      "lo95": P0 * np.exp(mu + q[0]), "lo80": P0 * np.exp(mu + q[1]),
                      "hi80": P0 * np.exp(mu + q[2]), "hi95": P0 * np.exp(mu + q[3])})
live = pd.DataFrame(live_rows)
live.to_csv(cfg.output_dir / "live_forecast.csv", index=False)
log_path = cfg.output_dir / "forecast_log.csv"
flog = pd.concat([pd.read_csv(log_path), live]) if log_path.exists() else live
flog.drop_duplicates(["asof", "h"], keep="last").to_csv(log_path, index=False)
display(live.set_index("h").round(2))

fig, ax = plt.subplots(figsize=(11, 4))
hist = price.iloc[-120:]
td = [d0] + [pd.Timestamp(t) for t in live["target_date"]]
col = lambda c: [P0] + list(live[c])
ax.plot(hist.index, hist, color="black", lw=1, label="Brent")
ax.plot(td, col("forecast_price"), color="#2E4A9E", lw=1.5, marker="o", ms=3, label="forecast")
ax.fill_between(td, col("lo80"), col("hi80"), color="#2E4A9E", alpha=0.25, label="80%")
ax.fill_between(td, col("lo95"), col("hi95"), color="#2E4A9E", alpha=0.12, label="95%")
ax.set_title(f"LASSO forecast from {d0.date()} (regime: {live['regime'].iloc[0]})", fontsize=11)
ax.set_ylabel("$/bbl"); ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.savefig(cfg.output_dir / "fan_chart.png", dpi=150); plt.show()

## 7. Takeaways

In [ ]:
o_l = lm[lm["sample"] == "Out-of-sample"].set_index("h")
i_l = lm[lm["sample"] == "In-sample"].set_index("h")
wins = int((o_l["RMSE_ratio_vs_RW"] < 1).sum())
sig_h = list(o_l.index[o_l["DM_p"] < 0.05])
nsel = (c20[[f"base:{n}" for n in names] + [f"turb:{n}" for n in names]] != 0).sum(axis=1)
lines = [
    f"Rolling window {cfg.window} days, re-estimated every {cfg.refit_every} days; the {cfg.plot_h}-day LASSO keeps a median of "
    f"{int(nsel.median())} of {2 * len(names)} terms (predictors and regime interactions); "
    f"{100 * (nsel == 0).mean():.0f}% of fits keep none.",
    f"Out-of-sample, the LASSO has lower price RMSE than RW at {wins} of {len(o_l)} horizons "
    f"(ratios {o_l['RMSE_ratio_vs_RW'].min():.3f} to {o_l['RMSE_ratio_vs_RW'].max():.3f}); "
    + (f"significant at 5% (DM) at h = {sig_h}." if sig_h else "none is significant at 5% (DM)."),
    f"In-sample ratios range from {i_l['RMSE_ratio_vs_RW'].min():.3f} to {i_l['RMSE_ratio_vs_RW'].max():.3f}; "
    "the gap to the out-of-sample ratios shows how much of the training fit does not carry over.",
    f"{len(spells)} regime spells since {lab.index[0].date()}; turbulent on {100 * lab.mean():.0f}% of days. "
    f"Regime-specific effects (gamma) are kept in {100 * (c20[[f'turb:{n}' for n in names]] != 0).any(axis=1).mean():.0f}% "
    f"of the {cfg.plot_h}-day fits.",
    f"Out-of-sample 95% interval coverage at 60 days: {o_l.loc[60, 'cover95_%']:.0f}%.",
]
display(Markdown("\n".join("- " + s for s in lines)))

## 8. Look-ahead check (optional)

Scrambles all data after a cut-off date, re-runs a short backtest, and checks that every forecast made on or before the cut-off is unchanged.

In [ ]:
if RUN_LEAK_CHECK:
    cut, end = pd.Timestamp("2021-06-30"), "2022-06-30"
    c2 = replace(cfg, oos_start="2020-06-01", horizons=(1, 20))
    rng = np.random.default_rng(0)
    p2, o2 = price.copy(), other.copy()
    after = p2.index > cut
    p2[after] = p2[after] * np.exp(rng.normal(0, 0.05, after.sum()))
    o2.loc[o2.index > cut] = o2.loc[o2.index > cut] * np.exp(rng.normal(0, 0.05, o2.loc[o2.index > cut].shape))
    res = []
    for pp_, oo_ in ((price, other), (p2, o2)):
        Xc, Yc, lc, vc = build_features(pp_.loc[:end], oo_.loc[:end], c2)
        res.append(run_backtest(Xc, Yc, lc, vc, c2, verbose=False))
    k = int(Xc.index.searchsorted(cut, side="right"))
    before = max(np.nanmax(np.abs(res[0][0][h][m][:k] - res[1][0][h][m][:k])) for h in c2.horizons
                 for m in ("LASSO", "q975", "AR", "HistMean"))
    before = max(before, np.nanmax(np.abs(res[0][3].to_numpy()[:k] - res[1][3].to_numpy()[:k])))
    later = np.nanmax(np.abs(res[0][0][20]["LASSO"][k:] - res[1][0][20]["LASSO"][k:]))
    print(f"max change on/before cut-off: {before:.2e}   after: {later:.2e}")
    print("PASS: no look-ahead" if before < 1e-9 and later > 0 else "FAIL")